In [1]:
!pip install qdrant-client sentence-transformers transformers torch

In [2]:
from qdrant_client import QdrantClient
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

/usr/local/lib/python3.10/dist-packages/sentence_transformers/cross_encoder/CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


In [3]:
# Qdrant 클라이언트 초기화
QDRANT_URL = "https://6e46b2c2-f28a-4f28-854d-432ab699fdfd.europe-west3-0.gcp.cloud.qdrant.io"
QDRANT_API_KEY = "u2eejPgTwIyhr7BVjFBtkjGdGYPWvzQTBkoYycErtm5cyrFjwEEH9w"
client = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY)


In [4]:
# 임베딩 모델 초기화
encoder = SentenceTransformer("jhgan/ko-sroberta-multitask")


/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [5]:
# ko-gpt-trinity-cw 모델 초기화
model_name = "centwon/ko-gpt-trinity-cw"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)


In [6]:
def search_disease_info(query_text, limit=3):
    query_vector = encoder.encode(query_text).tolist()
    search_results = client.search(
        collection_name='son5',
        query_vector=query_vector,
        limit=limit
    )
    return [{"질병": hit.payload.get("질병", "N/A"),
             "답변": hit.payload.get("답변", "N/A"),
             "유사도 점수": hit.score} for hit in search_results]


In [7]:
def generate_response(query, context):
    input_text = f"질문: {query}\n컨텍스트: {context}\n답변:"
    input_ids = tokenizer.encode(input_text, return_tensors="pt")

    with torch.no_grad():
        output = model.generate(
            input_ids,
            max_length=256,  # 응답 길이를 줄임
            num_return_sequences=1,
            no_repeat_ngram_size=2,
            top_k=50,
            top_p=0.92,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    return tokenizer.decode(output[0], skip_special_tokens=True).strip()


In [8]:
def advanced_rag_search():
    while True:
        query = input("질문을 입력하세요 (종료하려면 '끝내기' 입력): ")
        if query.lower() == '끝내기':
            print("검색을 종료합니다.")
            break

        results = search_disease_info(query)
        if results:
            best_match = results[0]
            print(f"\n'{query}'에 대한 검색 결과:\n")
            print(f"관련 질병: {best_match['질병']}")
            print(f"유사도 점수: {best_match['유사도 점수']:.4f}")
            print("\n생성된 답변:")
            generated_response = generate_response(query, best_match['답변'])
            print(generated_response)
        else:
            print("검색 결과가 없습니다.")
        print("-" * 50)

if __name__ == "__main__":
    advanced_rag_search()

질문을 입력하세요 (종료하려면 '끝내기' 입력): 운동 전후 종아리 뒤쪽 통증이 있고 발목을 움직일때 소리가 나

'운동 전후 종아리 뒤쪽 통증이 있고 발목을 움직일때 소리가 나'에 대한 검색 결과:

관련 질병: 아킬레스 건염
유사도 점수: 0.8145

생성된 답변:
질문: 운동 전후 종아리 뒤쪽 통증이 있고 발목을 움직일때 소리가 나
컨텍스트: 아킬레스 건염은 주로 발목 부근에서 발생하는 염증성 질환으로 알려져 있습니다. 아킬레스 건염은 발목 주변에서 통증과 부종을 동반하는 통증이 주로 나타납니다. 운동과 같은 활동 후 발뒤꿈치의 아킬레스 건 부분에서 염증이 발생하면 발목 주변에 강한 통증이 생길 수 있습니다. 또한, 아킬레스 건염은 종아리로 통증이 전해질 수 있어 발목 부분을 걷거나 달리는 동안 통증이 느껴질 수 있습니다. 만약 아킬레스 건염을 의심한다면, 휴식을 취해도 통증이 지속되는 경우 의사의 진료를 받는 것이 중요합니다. 아킬레스 건염은 다양한 증상을 동반하는 염증성 질환입니다. 만약 발목 주변에서 통증이 나타나면 의사의 진료를 받아야 합니다. 조기에 대처하지 않으면 심각한 합병증을 초래할 수 있으므로, 가능한 빠른 시일 내에 증상을 인지하고 의사의 도움을 받는 것이 중요합니다.
답변: 조급한 마음에 무리하게 운동을 하면 오히려 염증을 악화시킬 수 있고, 이로 인해 통증이 악화될 수 있으며 회복이 어려울 수 있으니 주의가 필요합니다. 또한, 무리한 운동은 증상을 악화시키고 재발의 원인이 될 수 있으니, 전문가의 조언과 지도가 필요합니다.
 관련 질병 : 아세트아미노펜(소아에 대한 사용 제한
--------------------------------------------------
질문을 입력하세요 (종료하려면 '끝내기' 입력): 통풍 예방 방법이 있을까?

'통풍 예방 방법이 있을까?'에 대한 검색 결과:

관련 질병: 통풍
유사도 점수: 0.7021

생성된 답변:
질문: 통풍 예방 방법이 있을까?
컨텍스트: 통풍성 관절염 예방을 위해서는 어떤

결과 :
 - 05 답변 시간 : 40~50초
 - 06 답변 시간 : 30~40초

약 10초 정도 답변 생성 시간을 단축 시킴